<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/SMS_messages_as_spam_or_not_spam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SMS Spam Classification using Deep Learning
This notebook demonstrates how to build a neural network to classify SMS messages as 'spam' or 'ham' (not spam).

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os

# Download the dataset from a reliable mirror
!wget --no-check-certificate https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-DataSet-/master/spam.csv -O SMSSpamCollection_raw.csv

# Note: This mirror uses a slightly different CSV format (comma separated with quotes)
# We will adjust the loading cell in the next step if necessary,
# but first let's ensure the file exists.

--2026-04-03 03:06:36--  https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-DataSet-/master/spam.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 503663 (492K) [application/octet-stream]
Saving to: ‘SMSSpamCollection_raw.csv’

SMSSpamCollection_r 100%[===================>] 491.86K  --.-KB/s    in 0.006s  

2026-04-03 03:06:36 (77.2 MB/s) - ‘SMSSpamCollection_raw.csv’ saved [503663/503663]



In [9]:
# Load and inspect data from the new source
df = pd.read_csv('SMSSpamCollection_raw.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
display(df.head())

# Encode labels: ham -> 0, spam -> 1
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

print(f"\nClass distribution:\n{df['label'].value_counts(normalize=True)}")

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



Class distribution:
label
0    0.865937
1    0.134063
Name: proportion, dtype: float64


In [10]:
# Text Preprocessing
max_words = 1000
max_len = 150

tok = Tokenizer(num_words=max_words)
tok.fit_on_texts(df['message'])

sequences = tok.texts_to_sequences(df['message'])
sequences_matrix = pad_sequences(sequences, maxlen=max_len)

X_train, X_test, Y_train, Y_test = train_test_split(sequences_matrix, df['label'], test_size=0.2, random_state=42)

In [11]:
# Build the Model
model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(max_words, 50, input_length=max_len),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Train the Model
history = model.fit(X_train, Y_train, batch_size=128, epochs=5, validation_split=0.2)

# Evaluate
accr = model.evaluate(X_test, Y_test)
print(f'Test set\n  Loss: {accr[0]:.3f}\n  Accuracy: {accr[1]:.3f}')

Epoch 1/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 14s 367ms/step - accuracy: 0.8564 - loss: 0.4263 - val_accuracy: 0.8823 - val_loss: 0.2643
Epoch 2/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 8s 263ms/step - accuracy: 0.9453 - loss: 0.1668 - val_accuracy: 0.9765 - val_loss: 0.1002
Epoch 3/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 11s 280ms/step - accuracy: 0.9851 - loss: 0.0646 - val_accuracy: 0.9809 - val_loss: 0.0653
Epoch 4/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 219ms/step - accuracy: 0.9893 - loss: 0.0402 - val_accuracy: 0.9821 - val_loss: 0.0611
Epoch 5/5
28/28 ━━━━━━━━━━━━━━━━━━━━ 10s 221ms/step - accuracy: 0.9933 - loss: 0.0259 - val_accuracy: 0.9821 - val_loss: 0.0619
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9794 - loss: 0.0746
Test set
  Loss: 0.075
  Accuracy: 0.979
